In [ ]:
from cellm import ask
from cellm.core import insert_cell

In [2]:
ask("Hi im Peter")

18:25:12 - LiteLLM:WARNING: utils.py:2730 - register_model: model=openrouter/z-ai/glm-5.2 not in built-in cost map and no prefix/region variant matched; cache cost fields will default to 0. To track cache cost, add cache_creation_input_token_cost and cache_read_input_token_cost to model_info


Hi Bob! How can I help? I'm ready to write code, debug, or explain concepts.

In [5]:
ask("Ahm, what, I said that i was Peter")

My apologies, Peter! The previous response was mistaken. How can I help you today?

You're right, my apologies Peter! In this notebook, `ask()` appears to be a custom function—likely an LLM wrapper—that executed and returned the greeting: "Hi Bob! How can I help? I'm ready to write code, debug, or explain concepts."

What would you like to work on?

You're right, my apologies Peter! I must have mixed up the names. How can I help you today?

Hi Peter! How can I help you today?

In [7]:
ask??

Signature: ask(pr: str)
Docstring: <no docstring>
Source:   
def ask(pr: str):
    cells_content = json.dumps(get_cells())
    chat.hist = [f"active notebook content: {cells_content}"]
    resp = chat(pr)
    chat.last_response = resp

    # display(Markdown(resp.choices[0].message.content))
    insert_cell(resp.choices[0].message.content)
File:      ~/code/nb_ask/nb_ask_py/src/cellm/core.py
Type:      function

In [9]:
insert_cell??

Signature: insert_cell(content)
Docstring: <no docstring>
Source:   
def insert_cell(content):
    return httpx.post(
        f"{EXT_URL}:{EXT_PORT}/insert_response",
        json={"content": content},
    )
File:      ~/code/nb_ask/nb_ask_py/src/cellm/core.py
Type:      function

I'm using the functions above to insert a llm response into the current notebook, as a markdown cell. Can you help me to improve the display?

In [10]:
ask("")

You can improve the display by parsing the LLM response to separate text and code blocks, then inserting them as markdown and code cells respectively. Here is a helper function to split the response:

```python
import re
from cellm.core import insert_cell

def insert_formatted_response(content: str):
    # Split content into markdown and code blocks
    parts = re.split(r'```python\n(.*?)```', content, flags=re.DOTALL)
    
    for i, part in enumerate(parts):
        if not part.strip():
            continue
        if i % 2 == 1:  # Odd indices are code blocks
            # Assuming your server can infer or you can extend it to accept cell type
            insert_cell(part.strip())
        else:  # Even indices are markdown
            insert_cell(part.strip())
```

If you want to keep it as markdown but improve rendering, you can display it directly in the output area instead of inserting a new cell:

```python
from IPython.display import display, Markdown

def ask(pr: str):
    cells_content = json.dumps(get_cells())
    chat.hist = [f"active notebook content: {cells_content}"]
    resp = chat(pr)
    chat.last_response = resp
    
    # Display beautifully in the output area
    display(Markdown(resp.choices[0].message.content))
```

In [11]:
ask("Im aware of the option to show it via `display(Markdown(…))` is there an option to make that output editable?")

No—`display(Markdown(...))` output is rendered and not editable. To get editable content, you need to insert it as an actual cell, which is what your current `insert_cell()` approach already does.

If you want the inserted cell to remain editable (rather than executed output), just ensure your server inserts it as a markdown cell (not code), which it appears to already do.

In [29]:
ask("Can we stell make the current markdown prettier, like wrap it in a markdown quote?")

Yes. Modify `ask` to wrap the response in a blockquote before inserting it:

```python
def ask(pr: str):
    cells_content = json.dumps(get_cells())
    chat.hist = [f"active notebook content: {cells_content}"]
    resp = chat(pr)
    chat.last_response = resp
    
    quoted = "> " + resp.choices[0].message.content.replace("\n", "\n> ")
    insert_cell(quoted)
```

In [17]:
import json
from cellm.core import chat

pr="Lets test that, give me 50 words lorem ipsum"
cells_content = json.dumps(get_cells())
chat.hist = [f"active notebook content: {cells_content}"]
resp = chat(pr)

In [27]:
from IPython.display import display, Markdown

print(resp.choices[0].message.content)

```python
ask("Give me 50 words lorem ipsum")
```


```python
ask("Give me 50 words lorem ipsum")
```

In [33]:
quoted = "> " + resp.choices[0].message.content.replace("\n", "\n> ")

In [36]:
display(Markdown(quoted))

> ```python
> ask("Give me 50 words lorem ipsum")
> ```

In [37]:
ask("Ok, that seems fine, i would also like to note that the cell is an llm output, can we have some header?")

Yes, you can add a markdown header before the blockquote to clearly label it as an LLM response:

```python
header = "**LLM Response:**\n\n"
quoted = header + "> " + resp.choices[0].message.content.replace("\n", "\n> ")
insert_cell(quoted)
```

In [39]:
content = chat.last_response.choices[0].message.content

In [43]:
header = "**LLM Response:**\n\n"
quoted = header + "> " + resp.choices[0].message.content.replace("\n", "\n> ")

In [44]:
display(Markdown(quoted))

**LLM Response:**

> ```python
> ask("Give me 50 words lorem ipsum")
> ```

In [45]:
ask("Hmm, ok, can we make it a bit mor 'pop'")

Use an emoji and a heading to make it stand out:

```python
header = "### 🤖 LLM Response\n\n> "
quoted = header + resp.choices[0].message.content.replace("\n", "\n> ")
insert_cell(quoted)
```

In [46]:
header = "### 🤖 LLM Response\n\n> "
quoted = header + resp.choices[0].message.content.replace("\n", "\n> ")

In [47]:
display(Markdown(quoted))

### 🤖 LLM Response

> ```python
> ask("Give me 50 words lorem ipsum")
> ```

In [48]:
ask??

Signature: ask(pr: str)
Docstring: <no docstring>
Source:   
def ask(pr: str):
    cells_content = json.dumps(get_cells())
    chat.hist = [f"active notebook content: {cells_content}"]
    resp = chat(pr)
    chat.last_response = resp

    # display(Markdown(resp.choices[0].message.content))
    insert_cell(resp.choices[0].message.content)
File:      ~/code/nb_ask/nb_ask_py/src/cellm/core.py
Type:      function

In [50]:
def ask(pr: str):
    cells_content = json.dumps(get_cells())
    chat.hist = [f"active notebook content: {cells_content}"]
    resp = chat(pr)
    chat.last_response = resp

    header = "### 🤖 LLM Response\n\n> "
    quoted = resp.choices[0].message.content.replace("\n", "\n> ")
    insert_cell(header+quoted)

In [51]:
ask("Whats the capital of France and give me a python function, thats adds one to any number")

### 🤖 LLM Response

> ### 🤖 LLM Response
> 
> > The capital of France is Paris.
> > 
> > ```python
> > def add_one(n):
> >     return n + 1
> > ```

In [52]:
ask("what happened here, why is there a douple 'LLM Response'?")

### 🤖 LLM Response

> The LLM sees previous outputs in the notebook context (like cells 28 and 30) containing "### 🤖 LLM Response" and mimics that formatting in its raw output. Then your `ask` function in cell 32 adds the header *again* when inserting the cell.
> 
> You can fix this by stripping any existing header from the LLM's response before quoting it:
> 
> ```python
> def ask(pr: str):
>     cells_content = json.dumps(get_cells())
>     chat.hist = [f"active notebook content: {cells_content}"]
>     resp = chat(pr)
>     chat.last_response = resp
> 
>     header = "### 🤖 LLM Response\n\n> "
>     content = resp.choices[0].message.content.replace("### 🤖 LLM Response\n\n", "")
>     quoted = content.replace("\n", "\n> ")
>     insert_cell(header + quoted)
> ```

In [53]:
ask("Would it help if I add something in regard to that to the system prompt?")

### 🤖 LLM Response

> Yes. Add an instruction to output only the raw response without adding headers or mimicking previous cells' formatting.

In [54]:
ask("Make a suggestion for a good and concise system prompt to add")

### 🤖 LLM Response

> ```text
> You are an AI assistant. Output only the raw response without any headers, blockquotes, or markdown wrappers like "### 🤖 LLM Response". Do not mimic formatting from previous cells.
> ```

In [55]:
ask("ah, does the `chat` instance know what llm model is used?")

### 🤖 LLM Response

> Check `chat.model` or `chat??` to see its attributes.

In [63]:
ask("how do i get the used input and output tokens from `chat`?")

## 🤖 LLM Response (glm-5.2)

> Here is some output for you!

#### 🤖 LLM Response (glm-5.2)

> Check the `.usage` attribute on the response object:
> 
> ```python
> resp = chat.last_response
> input_tokens = resp.usage.prompt_tokens
> output_tokens = resp.usage.completion_tokens
> ```

In [67]:
ask("give me a random number from 1 to 65465498764")

##### 🤖 LLM Response (glm-5.2 | in: 4260 | out: 33)

> 42873654129

In [68]:
ask("So those 11 digits require 33 completion tokens?")

## 🤖 LLM Response (glm-5.2 | in: 4359 | out: 597)

> Not necessarily. Tokenizers vary—GLM's may split numbers into 2-3 digit chunks, and the raw response likely included extra text (reasoning, explanation) that was trimmed in the displayed output.